# 📰 HIN-GNN News Recommendation System
### MIND-Small Dataset · Heterogeneous Information Network · Graph Neural Networks

---
**Architecture Overview:**
```
 MIND-Small Dataset
       ↓
 ┌─────────────────────────────────────────┐
 │  Heterogeneous Information Network (HIN)│
 │                                         │
 │  Users ──click──▶ News ──belongs──▶ Category │
 │                    │                    │
 │                    └──tagged──▶ Entity  │
 │                    (Wikidata KG nodes)  │
 └─────────────────────────────────────────┘
       ↓
 ┌─────────────┐  ┌──────────────────────┐
 │  HIN-GNN    │  │  HAN                 │
 │ (unified    │  │ (per-relation attn   │
 │  LightGCN)  │  │  Heterogeneous Attn) │
 └─────────────┘  └──────────────────────┘
       ↓
 News Recommendations  +  Diversity Re-ranking (MMR)
```

## 1. Setup & Imports

In [ ]:
import os, sys, json, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from collections import defaultdict

# Project root
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = os.path.join(ROOT, 'data')
ARTIFACTS_DIR = os.path.join(ROOT, 'artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')
print(f'Root    : {ROOT}')

## 2. Generate MIND-Small Dataset

In [ ]:
from data.mind_small_generator import generate_mind_small_dataset, ARTICLES, CAT_BUCKETS

generate_mind_small_dataset(DATA_DIR, n_train_users=400, n_dev_users=100)

# Quick stats
cat_counts = defaultdict(int)
for a in ARTICLES:
    cat_counts[a['category']] += 1

print(f'\nDataset summary:')
print(f'  Total articles : {len(ARTICLES):,}')
print(f'  Categories     : {len(cat_counts)}')
print(f'  Category breakdown:')
for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f'    {cat:<20s} {cnt:>4d} articles')

In [ ]:
# Visualize category distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cats = list(cat_counts.keys())
vals = [cat_counts[c] for c in cats]
colors = plt.cm.Set3(np.linspace(0, 1, len(cats)))

axes[0].bar(cats, vals, color=colors, edgecolor='gray')
axes[0].set_title('Article Count per Category', fontsize=13)
axes[0].set_xlabel('Category'); axes[0].set_ylabel('# Articles')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.4)

axes[1].pie(vals, labels=cats, colors=colors, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Category Distribution', fontsize=13)

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'category_distribution.png'), dpi=150)
plt.show()

## 3. Load & Explore Data

In [ ]:
from src.data_loader import load_news, load_behaviors, extract_training_triplets

train_news = load_news(os.path.join(DATA_DIR, 'train', 'news.tsv'))
dev_news   = load_news(os.path.join(DATA_DIR, 'dev',   'news.tsv'))
news_df    = pd.concat([train_news, dev_news]).drop_duplicates(subset=['news_id']).reset_index(drop=True)

train_beh = load_behaviors(os.path.join(DATA_DIR, 'train', 'behaviors.tsv'))
dev_beh   = load_behaviors(os.path.join(DATA_DIR, 'dev',   'behaviors.tsv'))

news_ids = set(news_df['news_id'])
triplets, pop_counts = extract_training_triplets(train_beh, news_ids)

user_ids = sorted(set(train_beh['user_id']))
user_map = {u: i for i, u in enumerate(user_ids)}
news_map = {n: i for i, n in enumerate(news_df['news_id'])}
n_users  = len(user_map)
n_news   = len(news_map)

print(f'News articles : {len(news_df):,}')
print(f'Train users   : {n_users:,}')
print(f'Train rows    : {len(train_beh):,}')
print(f'Dev rows      : {len(dev_beh):,}')
print(f'BPR triplets  : {len(triplets):,}')

news_df.head(3)

## 4. Build Heterogeneous Information Network (HIN)

In [ ]:
from src.graph_builder import construct_hin, build_sparse_adjacency, build_relation_adjacencies

print('Constructing HIN...')
hin_graph, edge_counts = construct_hin(news_df, train_beh)

# Inspect node types
type_counts = defaultdict(int)
for _, d in hin_graph.nodes(data=True):
    type_counts[d.get('node_type', 'unknown')] += 1

print(f'\nHIN Statistics:')
print(f'  Total nodes: {hin_graph.number_of_nodes():,}')
print(f'  Total edges: {hin_graph.number_of_edges():,}')
print(f'  Node types :')
for nt, cnt in sorted(type_counts.items()):
    print(f'    {nt:<15s}: {cnt:>5,}')
print(f'  Edge types :')
for et, cnt in sorted(edge_counts.items()):
    print(f'    {et:<20s}: {cnt:>5,}')

In [ ]:
# Visualize a small subgraph of the HIN
sample_news = list(news_df['news_id'])[:8]
sub_nodes = set(sample_news)

# Add neighbors of sample news
for nn in sample_news:
    for neighbor in hin_graph.neighbors(nn):
        sub_nodes.add(neighbor)

sub_nodes = list(sub_nodes)[:60]  # cap
subG = hin_graph.subgraph(sub_nodes)

type_colors = {
    'news': '#4e9af1',
    'user': '#f1c94e',
    'category': '#f17c4e',
    'subcategory': '#a3f14e',
    'entity': '#c44ef1',
}
node_colors = [type_colors.get(subG.nodes[n].get('node_type','news'), '#cccccc') for n in subG.nodes]

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(subG, seed=42, k=0.6)
nx.draw_networkx(subG, pos, node_color=node_colors, node_size=200,
                 with_labels=False, alpha=0.85, edge_color='#aaaaaa', width=0.6)

legend = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
plt.legend(handles=legend, fontsize=10, loc='upper left')
plt.title('HIN Subgraph — Users, News, Categories, Subcategories, Entities', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'hin_subgraph.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Build adjacency matrices
print('Building adjacency matrices...')
adj_full = build_sparse_adjacency(user_map, news_map, triplets, news_df, hin_graph, DEVICE)
rel_adjs = build_relation_adjacencies(user_map, news_map, triplets, news_df, DEVICE)
RELATION_NAMES = list(rel_adjs.keys())
print(f'\nRelation types: {RELATION_NAMES}')

## 5. Train HIN-GNN and HAN Models

In [ ]:
from src.models import HINGNN, HAN, train_gnn_model, train_han_model

EMBED_DIM = 64
EPOCHS    = 8
total_nodes = n_users + n_news

print('=' * 60)
print('Training HIN-GNN (LightGCN-style on unified HIN adjacency)...')
print('=' * 60)
hin_model = HINGNN(total_nodes, n_users, n_news, embedding_dim=EMBED_DIM, num_layers=3).to(DEVICE)
hin_emb   = train_gnn_model(hin_model, adj_full, triplets, user_map, news_map,
                             epochs=EPOCHS, lr=0.005, device=DEVICE)
print(f'HIN-GNN embedding shape: {hin_emb.shape}')

In [ ]:
print('=' * 60)
print('Training HAN (Heterogeneous Attention Network)...')
print('=' * 60)
han_model = HAN(total_nodes, n_users, n_news, embedding_dim=EMBED_DIM, num_layers=2,
                relation_names=RELATION_NAMES, dropout=0.1).to(DEVICE)
han_emb   = train_han_model(han_model, rel_adjs, triplets, user_map, news_map,
                             epochs=EPOCHS, lr=0.005, device=DEVICE)
print(f'HAN embedding shape: {han_emb.shape}')

In [ ]:
# Inspect HAN attention weights
import torch.nn.functional as F
han_model.eval()
with torch.no_grad():
    attn_weights = F.softmax(han_model.layers[0].attn, dim=0).cpu().numpy()

print('HAN Semantic Attention Weights (relation importance):')
for rel, w in zip(RELATION_NAMES, attn_weights):
    bar = '█' * int(w * 50)
    print(f'  {rel:<22s}: {w:.4f} {bar}')

plt.figure(figsize=(8, 4))
plt.bar(RELATION_NAMES, attn_weights, color=['#4e9af1','#f1c94e','#f17c4e','#a3f14e'][:len(RELATION_NAMES)])
plt.title('HAN Semantic Attention Weights per Relation Type', fontsize=12)
plt.xlabel('Relation Type'); plt.ylabel('Attention Weight')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'han_attention.png'), dpi=150)
plt.show()

## 6. Train Baselines

In [ ]:
from src.baselines import (
    PopularityRecommender, TFIDFRecommender,
    MatrixFactorizationSVD, train_lightgcn_baseline
)

print('Setting up baselines...')

# 1. Popularity
pop_rec = PopularityRecommender(pop_counts)
print('  ✓ Popularity recommender ready')

# 2. TF-IDF
tfidf_rec = TFIDFRecommender(news_df)
tfidf_rec.build_user_profiles(train_beh)
print('  ✓ TF-IDF content profiles built')

# 3. SVD Matrix Factorization
svd_rec = MatrixFactorizationSVD(user_map, news_map, triplets, n_components=EMBED_DIM)
print('  ✓ SVD Matrix Factorization fitted')

# 4. LightGCN (CF-only)
lgcn_emb = train_lightgcn_baseline(user_map, news_map, triplets,
                                    epochs=EPOCHS, embedding_dim=EMBED_DIM, device=DEVICE)

## 7. Generate Sample Recommendations

In [ ]:
def emb_score(emb, u, n):
    if u in user_map and n in news_map:
        return float(np.dot(emb[user_map[u]], emb[n_users + news_map[n]]))
    return 0.0

news_meta = news_df.set_index('news_id').to_dict(orient='index')

def show_recommendations(user_id, model_emb=None, model_name='HIN-GNN', top_k=5):
    if model_emb is not None:
        all_news = list(news_map.keys())
        scores = {n: emb_score(model_emb, user_id, n) for n in all_news if n in news_map}
        ranked = sorted(scores, key=scores.get, reverse=True)[:top_k]
    else:
        ranked = pop_rec.rank(user_id, list(news_map.keys()))[:top_k]
    
    print(f"\n📰 Top-{top_k} Recommendations for User '{user_id}' [{model_name}]:")
    for i, nid in enumerate(ranked, 1):
        meta = news_meta.get(nid, {})
        print(f"  {i}. [{meta.get('category','?')}/{meta.get('subcategory','?')}] {meta.get('title','?')[:70]}")

sample_user = list(user_map.keys())[0]
show_recommendations(sample_user, hin_emb, 'HIN-GNN')
show_recommendations(sample_user, han_emb, 'HAN')
show_recommendations(sample_user, lgcn_emb, 'LightGCN (CF-only)')
show_recommendations(sample_user, None, 'Popularity')

## 8. Evaluate All Models

In [ ]:
from src.metrics import compute_ranking_accuracy, compute_novelty_and_diversity
from src.data_loader import parse_impressions

TOP_K = 10

def evaluate(name, predict_fn, k=TOP_K):
    acc_rows, div_rows = [], []
    for row in dev_beh.itertuples(index=False):
        parsed     = parse_impressions(row.impressions)
        candidates = [nid for nid, _ in parsed if nid in news_map]
        relevant   = [nid for nid, lbl in parsed if lbl == 1 and nid in news_map]
        if not candidates or not relevant:
            continue
        scores = {nid: predict_fn(row.user_id, nid) for nid in candidates}
        ranked = sorted(candidates, key=lambda n: scores.get(n,-1e9), reverse=True)[:k]
        acc = compute_ranking_accuracy(ranked, relevant, k=k)
        if acc:
            div = compute_novelty_and_diversity(ranked, news_df, pop_counts, k=k)
            acc_rows.append(acc); div_rows.append(div)
    if not acc_rows:
        return {'Model': name, f'nDCG@{k}': 0.0, 'MRR': 0.0, f'Recall@{k}': 0.0,
                'Novelty': 0.0, 'CategoryDiversity': 0.0, 'ILD': 0.0}
    return {'Model': name, **pd.DataFrame(acc_rows).mean().to_dict(),
                           **pd.DataFrame(div_rows).mean().to_dict()}

print(f'Evaluating 6 models on {len(dev_beh):,} dev impressions (top-{TOP_K})...')
print('-' * 70)

results = [
    evaluate('Popularity',          pop_rec.predict_score),
    evaluate('TF-IDF Content',      tfidf_rec.predict_score),
    evaluate('SVD (MF)',            svd_rec.predict_score),
    evaluate('LightGCN (CF-only)',  lambda u,n: emb_score(lgcn_emb, u, n)),
    evaluate('HIN-GNN (Proposed)',  lambda u,n: emb_score(hin_emb,  u, n)),
    evaluate('HAN (Proposed)',      lambda u,n: emb_score(han_emb,  u, n)),
]

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(ARTIFACTS_DIR, 'results.csv'), index=False)

display(results_df.set_index('Model').style
        .background_gradient(cmap='YlGn', subset=[f'nDCG@{TOP_K}','MRR',f'Recall@{TOP_K}'])
        .background_gradient(cmap='YlOrBr', subset=['ILD','CategoryDiversity'])
        .format('{:.4f}'))

## 9. Comparison Charts

In [ ]:
# Multi-metric grouped bar chart
metrics = [f'nDCG@{TOP_K}', 'MRR', f'Recall@{TOP_K}', 'ILD', 'CategoryDiversity']
models  = results_df['Model'].tolist()
x       = np.arange(len(models))
bar_w   = 0.15
colors  = plt.cm.Set2(np.linspace(0, 1, len(metrics)))

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = results_df[metric].tolist()
    ax.bar(x + i * bar_w, vals, bar_w, label=metric, color=color, edgecolor='gray', alpha=0.9)

ax.set_xticks(x + bar_w * (len(metrics)-1) / 2)
ax.set_xticklabels(models, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('All Models — Multi-Metric Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left', ncol=len(metrics))
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'comparison_grouped.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Radar chart comparison
from matplotlib.patches import FancyArrowPatch
import matplotlib

radar_metrics = [f'nDCG@{TOP_K}', 'MRR', f'Recall@{TOP_K}', 'Novelty', 'ILD', 'CategoryDiversity']
N = len(radar_metrics)
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
radar_colors = ['#aaaaaa', '#ffd700', '#ff8c00', '#1e90ff', '#32cd32', '#ff1493']

for idx, row in results_df.iterrows():
    vals_raw = [row[m] for m in radar_metrics]
    # Normalize to [0,1] relative to max across models
    maxes = results_df[radar_metrics].max().values
    vals_norm = [v / (m + 1e-9) for v, m in zip(vals_raw, maxes)]
    vals_norm += vals_norm[:1]
    ax.plot(angles, vals_norm, 'o-', lw=2, label=row['Model'], color=radar_colors[idx])
    ax.fill(angles, vals_norm, alpha=0.1, color=radar_colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=10)
ax.set_title('Model Comparison Radar Chart\n(normalized to max per metric)', fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'radar_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Accuracy vs Diversity Trade-off (MMR Re-ranking)

In [ ]:
from src.reranker import diversify_recommendations

alphas = [1.0, 0.8, 0.6, 0.4, 0.2]
tradeoff = []

for alpha in alphas:
    acc_l, div_l = [], []
    for row in dev_beh.itertuples(index=False):
        parsed     = parse_impressions(row.impressions)
        candidates = [nid for nid, _ in parsed if nid in news_map]
        relevant   = [nid for nid, lbl in parsed if lbl == 1 and nid in news_map]
        if not candidates or not relevant: continue
        scores   = {nid: emb_score(han_emb, row.user_id, nid) for nid in candidates}
        reranked = diversify_recommendations(candidates, scores, news_df, alpha=alpha, top_k=TOP_K)
        acc = compute_ranking_accuracy(reranked, relevant, k=TOP_K)
        if acc:
            div = compute_novelty_and_diversity(reranked, news_df, pop_counts, k=TOP_K)
            acc_l.append(acc); div_l.append(div)
    if acc_l:
        r = {'Alpha': alpha, **pd.DataFrame(acc_l).mean().to_dict(),
                             **pd.DataFrame(div_l).mean().to_dict()}
        tradeoff.append(r)
        print(f"  α={alpha:.1f} → nDCG@10={r[f'nDCG@{TOP_K}']:.4f}  ILD={r['ILD']:.4f}  CatDiv={r['CategoryDiversity']:.4f}")

tradeoff_df = pd.DataFrame(tradeoff)
tradeoff_df.to_csv(os.path.join(ARTIFACTS_DIR, 'diversity_tradeoff.csv'), index=False)
print('\nTrade-off analysis saved.')

In [ ]:
# Trade-off plot
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.plot(tradeoff_df['Alpha'], tradeoff_df[f'nDCG@{TOP_K}'], 'b-o', lw=2.5, ms=8, label=f'nDCG@{TOP_K} (accuracy)')
ax1.plot(tradeoff_df['Alpha'], tradeoff_df['MRR'], 'b--s', lw=1.5, ms=6, alpha=0.7, label='MRR')
ax2.plot(tradeoff_df['Alpha'], tradeoff_df['ILD'], 'g-^', lw=2.5, ms=8, label='ILD (diversity)')
ax2.plot(tradeoff_df['Alpha'], tradeoff_df['CategoryDiversity'], 'g--D', lw=1.5, ms=6, alpha=0.7, label='CatDiversity')

ax1.set_xlabel('Relevance-Diversity Balance (α)', fontsize=12)
ax1.set_ylabel('Accuracy Metrics', color='blue', fontsize=11)
ax2.set_ylabel('Diversity Metrics', color='green', fontsize=11)
ax1.tick_params(axis='y', labelcolor='blue')
ax2.tick_params(axis='y', labelcolor='green')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='center right', fontsize=9)

plt.title('HAN: Accuracy vs Diversity Trade-off (MMR Re-ranking)', fontsize=13, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.4)
plt.annotate('← Diversity-focused', xy=(0.25, 0.08), xycoords='axes fraction', fontsize=9, color='green', alpha=0.8)
plt.annotate('Accuracy-focused →', xy=(0.65, 0.08), xycoords='axes fraction', fontsize=9, color='blue', alpha=0.8)
fig.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'tradeoff_plot.png'), dpi=200)
plt.show()

## 11. Embedding Visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

# t-SNE on news embeddings (HIN-GNN), colored by category
news_embs = hin_emb[n_users:]   # (n_news, dim)
news_cats = news_df['category'].tolist()
unique_cats = sorted(set(news_cats))
cat_to_idx  = {c: i for i, c in enumerate(unique_cats)}
cat_indices = [cat_to_idx[c] for c in news_cats]

print('Running t-SNE on news embeddings...')
tsne = TSNE(n_components=2, perplexity=min(30, len(news_embs)-1), random_state=SEED, n_iter=1000)
emb_2d = tsne.fit_transform(news_embs[:500])  # sample for speed
cats_sample = news_cats[:500]

palette = plt.cm.tab10(np.linspace(0, 1, len(unique_cats)))
plt.figure(figsize=(12, 8))
for i, cat in enumerate(unique_cats):
    mask = [c == cat for c in cats_sample]
    pts  = emb_2d[mask]
    plt.scatter(pts[:, 0], pts[:, 1], c=[palette[i]], label=cat, alpha=0.7, s=25, edgecolors='none')

plt.legend(fontsize=9, loc='best', markerscale=1.5, ncol=2)
plt.title('t-SNE of HIN-GNN News Embeddings (colored by category)', fontsize=13)
plt.xlabel('t-SNE dim 1'); plt.ylabel('t-SNE dim 2')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'tsne_embeddings.png'), dpi=150)
plt.show()

## 12. Final Summary

In [ ]:
print('\n' + '='*70)
print('  FINAL RESULTS SUMMARY')
print('='*70)
display(results_df.set_index('Model').round(4))

# Which model wins?
best_ndcg = results_df.loc[results_df[f'nDCG@{TOP_K}'].idxmax(), 'Model']
best_div  = results_df.loc[results_df['ILD'].idxmax(), 'Model']
print(f'\n  Best nDCG@{TOP_K} : {best_ndcg}')
print(f'  Best ILD (Div) : {best_div}')
print('\n  Artifacts saved in:', ARTIFACTS_DIR)
print('='*70)